# 1. Install & Import

In [ ]:
!pip install datasets sacrebleu -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import numpy as np
import re, random, time, math
from collections import Counter
import sacrebleu
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Load Dataset — Tatoeba (ar → en)

> **Why Tatoeba?**  
> opus_books is literary text — long, complex, varied. Tatoeba contains short everyday sentences
> ("Where is the station?", "I like coffee") that are much easier for an LSTM to learn from.
> This alone will significantly reduce your loss.

In [ ]:
from datasets import load_dataset

# Fallback: opus100 ar-en (standard Parquet format, no script needed)
dataset = load_dataset('Helsinki-NLP/opus-100', 'ar-en')
print(dataset)

all_data = []
for split in dataset:
    for ex in dataset[split]:
        all_data.append((ex['translation']['ar'], ex['translation']['en']))

print(f'\nTotal pairs: {len(all_data)}')
print('Sample:', all_data[0])

In [ ]:
# Keep only 80k pairs — enough to learn, fast enough to train
random.shuffle(all_data)
all_data = all_data[:80_000]
print(f'Using: {len(all_data)} pairs')


## 3. Preprocessing

In [ ]:
def normalize_arabic(text):
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)   # remove diacritics
    text = re.sub(r'[\u0622\u0623\u0625\u0671]', '\u0627', text) # normalize alef
    text = re.sub(r'\u0629', '\u0647', text)                     # ta marbuta → ha
    text = re.sub(r'\u0649', '\u064A', text)                     # alef maqsura → ya
    text = re.sub(r'\u0640', '', text)                           # remove tatweel
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)             # keep Arabic only
    return re.sub(r'\s+', ' ', text).strip()

def normalize_english(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def tokenize(text):
    return text.split()

# --- Filter: keep only short, clean pairs ---
MAX_LEN   = 20   # tighter than v1 (was 30) → cleaner pairs
MIN_LEN   = 2

pairs = []
for ar_raw, en_raw in all_data:
    ar_tok = tokenize(normalize_arabic(ar_raw))
    en_tok = tokenize(normalize_english(en_raw))
    if MIN_LEN <= len(ar_tok) <= MAX_LEN and MIN_LEN <= len(en_tok) <= MAX_LEN:
        pairs.append((ar_tok, en_tok))

random.shuffle(pairs)
print(f'Usable pairs after filtering: {len(pairs)}')

# Print length distribution
ar_lens = [len(p[0]) for p in pairs]
en_lens = [len(p[1]) for p in pairs]
print(f'AR avg length: {np.mean(ar_lens):.1f} | EN avg length: {np.mean(en_lens):.1f}')

# Show some examples
for ar, en in pairs[:5]:
    print(f'  AR: {" ".join(ar)}')
    print(f'  EN: {" ".join(en)}')
    print()

## 4. Vocabulary

In [ ]:
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3
SPECIAL = ['<pad>', '<sos>', '<eos>', '<unk>']

class Vocabulary:
    def __init__(self, min_freq=2):
        self.min_freq  = min_freq
        self.word2idx  = {t: i for i, t in enumerate(SPECIAL)}
        self.idx2word  = {i: t for t, i in self.word2idx.items()}

    def build(self, token_lists):
        counter = Counter(tok for toks in token_lists for tok in toks)
        for word, freq in sorted(counter.items(), key=lambda x: -x[1]):
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word

    def encode(self, tokens):
        return [self.word2idx.get(t, UNK_IDX) for t in tokens]

    def decode(self, indices):
        words = []
        for i in indices:
            w = self.idx2word.get(i, '<unk>')
            if w in ('<eos>', '<pad>'): break
            if w not in ('<sos>',): words.append(w)
        return words

    def __len__(self): return len(self.word2idx)

ar_vocab = Vocabulary(min_freq=2)
en_vocab = Vocabulary(min_freq=2)

ar_vocab.build([p[0] for p in pairs])
en_vocab.build([p[1] for p in pairs])

print(f'Arabic  vocab: {len(ar_vocab):,}')
print(f'English vocab: {len(en_vocab):,}')